# Create dataset for card strength training

1. Get all available cards
2. Search for labels in json files
3. create features

In [1]:
import os
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import QuantileTransformer
import re
from sentence_transformers import SentenceTransformer

from edhrec_card_scraper import sluggify_card_name
from src.dataset_filter import filter_dataset


C:\Users\ruflu\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
INPUT_DATA_PATH = "data/scryfall/oracle-cards-20251022090300.json"
OUTPUT_DATA_PATH = "data/card_strength.xlsx"
EDHREC_CARD_DIRECTORY = "data/edhrec/cards"

In [3]:
df = pd.read_json(INPUT_DATA_PATH)
df = filter_dataset(df)

print(f"shape: {df.shape}")
df.head()

Loading dataset...
Cleaning text encodings...
Dataset has been filtered and cleaned.
shape: (29677, 14)


,oracle_id,name,card_faces,mana_cost,cmc,type_line,oracle_text,power,toughness,loyalty,colors,color_identity,keywords,game_changer
0,00037840-6089-42ec-8c5c-281f9f474504,"Nissa, Worldsoul Speaker",NaN,{3}{G},4.0,Legendary Creature — Elf Druid,"Landfall — Whenever a land you control enters,...",3,3,NaN,[G],[G],[Landfall],False
1,0004ebd0-dfd6-4276-b4a6-de0003e94237,Static Orb,NaN,{3},3.0,Artifact,"As long as this artifact is untapped, players ...",NaN,NaN,NaN,[],[],[],False
2,0006faf6-7a61-426c-9034-579f2cfcfa83,Sensory Deprivation,NaN,{U},1.0,Enchantment — Aura,Enchant creature\nEnchanted creature gets -3/-0.,NaN,NaN,NaN,[U],[U],[Enchant],False
3,0007c283-5b7a-4c00-9ca1-b455c8dff8c3,Road of Return,NaN,{G}{G},2.0,Sorcery,Choose one —\n• Return target permanent card f...,NaN,NaN,NaN,[G],[G],[Entwine],False
4,000d5588-5a4c-434e-988d-396632ade42c,Storm Crow,NaN,{1}{U},2.0,Creature — Bird,Flying (This creature can't be blocked except ...,1,2,NaN,[U],[U],[Flying],False


## Handle multi-face cards

In [13]:
def prepare_transformer_input(row):
    faces_text = []
    
    # 1. Check if it's a multi-face card (card_faces exists and is a list)
    # Using row.get() safely checks if the column exists at all
    card_faces = row.get('card_faces')
    
    if isinstance(card_faces, list) and len(card_faces) > 0:
        # Loop through each face (e.g., Heaven, then Earth)
        for face in card_faces:
            face_name = str(face.get('name', ''))
            face_type = str(face.get('type_line', ''))
            face_oracle = str(face.get('oracle_text', ''))
            
            # Clean reminder text
            cleaned_oracle = re.sub(r'\(.*?\)', '', face_oracle)
            
            # Replace the specific face's name with CARD_NAME
            if face_name:
                cleaned_oracle = cleaned_oracle.replace(face_name, "CARD_NAME")
                
            cleaned_oracle = re.sub(r'\s+', ' ', cleaned_oracle).strip()
            
            # Append this face's formatted string to our list
            faces_text.append(f"Type: {face_type}. Effect: {cleaned_oracle}")
            
        # Join the faces together with a logical separator
        final_string = " // ".join(faces_text)
        
    # 2. Handle normal, single-face cards
    else:
        name = str(row['name'])
        type_line = str(row['type_line'])
        oracle_text = str(row['oracle_text']) if pd.notna(row['oracle_text']) else ""
        
        # Clean reminder text
        cleaned_oracle = re.sub(r'\(.*?\)', '', oracle_text)
        
        # Replace the card's name
        cleaned_oracle = cleaned_oracle.replace(name, "CARD_NAME")
        cleaned_oracle = re.sub(r'\s+', ' ', cleaned_oracle).strip()
        
        final_string = f"Type: {type_line}. Effect: {cleaned_oracle}"
        
    return final_string

df['transformer_input'] = df.apply(prepare_transformer_input, axis=1)

## Fill label columns from card data

In [ ]:
# Lists to store our new data and the rows we need to delete
new_data = []
indices_to_drop = []

for idx, row in df.iterrows():
    name = row["name"]

    if name in ["Forest", "Mountain", "Swamp", "Island", "Plains"]:
        print(f"Info: remove basic land.")
        indices_to_drop.append(idx)
        continue

    slugs = sluggify_card_name(name)
    
    # Bulletproof check: if slugs is somehow a string, make it a list
    if isinstance(slugs, str):
        slugs = [slugs]

    # Initialize values
    inclusion = potential_decks = inclusion_rate = log_inclusion_rate = 0

    for slug in slugs:
        filename = f"{slug}.json"
        path = os.path.join(EDHREC_CARD_DIRECTORY, filename)

        if not os.path.exists(path):
            continue

        # Read json
        try:
            with open(path, 'r', encoding='utf-8') as f:
                json_dict = json.load(f)
                
            # Navigate your JSON structure
            inclusion = json_dict["container"]["json_dict"]["card"]["inclusion"]
            potential_decks = json_dict["container"]["json_dict"]["card"]["potential_decks"]
            
            # Enforce 64-bit float precision for the division
            inclusion_rate = np.float64(inclusion) / np.float64(potential_decks)
            
            # Use log1p for mathematically flawless ln(1+x) calculation on small numbers
            log_inclusion_rate = np.log10(inclusion_rate)
            
            # SUCCESS! We found valid data. Break out of the 'for slug in slugs:' loop!
            break
            
        except (KeyError, json.JSONDecodeError):
            # This handles the "redirect" files perfectly. It fails to find the keys, 
            # ignores this file, and continues to the next slug in the list.
            continue

    # check if data was found
    if inclusion == 0:
        print(f"Error: no inclusion data found for {name} using: {slugs}\nDropping row.")
        indices_to_drop.append(idx)
        continue

    # Save the calculated values tied to the original index
    new_data.append({
        'index': idx,
        'inclusion': inclusion,
        'potential_decks': potential_decks,
        'inclusion_rate': inclusion_rate,
        'log_inclusion_rate': log_inclusion_rate
    })

# 3. Drop the missing/broken rows all at once
df = df.drop(index=indices_to_drop)

# 4. Convert our collected data into a DataFrame and merge it back into the original
results_df = pd.DataFrame(new_data).set_index('index')
df = df.join(results_df)

print(f"Processing complete. {len(indices_to_drop)} rows dropped. Current DataFrame shape: {df.shape}")

## Create power scores

In [ ]:
# --- TRUE LOGARITHMIC SCALING ---
MIN_SCORE = 25
MAX_SCORE = 99

log_min = df['log_inclusion_rate'].min()
log_max = df['log_inclusion_rate'].max()

# Apply the Min-Max formula to map it strictly between 25.0 and 99.0
df['log10_power_score'] = MIN_SCORE + ((df['log_inclusion_rate'] - log_min) * (MAX_SCORE - MIN_SCORE)) / (log_max - log_min)

print(f"Scores calculated! Range: {df['log10_power_score'].min():.4f} to {df['log10_power_score'].max():.4f}")

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))
sns.histplot(df['gauss_power_score'], bins=70, kde=True, color='indigo')

plt.title('Distribution of MTG Card Power Scores (30-99)', fontsize=16, fontweight='bold')
plt.xlabel('Calculated Power Score', fontsize=12)
plt.ylabel('Number of Cards', fontsize=12)
plt.xlim(25, 105)
plt.show()

## Create text embeddings

In [14]:
print("Loading all-MiniLM-L6-v2...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating 384-dimensional embeddings...")
embeddings = model.encode(
    df['transformer_input'].tolist(), 
    batch_size=256, 
    show_progress_bar=True,
    convert_to_numpy=True 
)

print(f"Text Embeddings generated! Shape: {embeddings.shape}")

Loading all-MiniLM-L6-v2...
Generating 384-dimensional embeddings...


Batches: 100%|██████████| 116/116 [03:58<00:00,  2.06s/it]

Text Embeddings generated! Shape: (29677, 384)


## Create numerical and boolean features

In [ ]:
# 3. Clean and Extract Numerical Features
# MTG power/toughness can be weird strings like "1+*" or "*". We need to safely convert them to floats.
def safe_to_float(val):
    if pd.isna(val):
        return np.nan
    val = str(val).replace('*', '0').replace('+', '') # Treat * as 0
    try:
        return float(val)
    except ValueError:
        return np.nan

print("Processing tabular features...")
# Create safe numerical columns
df['num_cmc'] = pd.to_numeric(df['cmc'], errors='coerce').fillna(0)
df['num_power'] = df['power'].apply(safe_to_float)
df['num_toughness'] = df['toughness'].apply(safe_to_float)
df['num_loyalty'] = df['loyalty'].apply(safe_to_float) # Will be NaN for non-planeswalkers

# Extract those columns into a numpy array
# We use np.nan_to_num to replace NaNs with an out-of-bounds value (-999) 
# so XGBoost explicitly knows it's missing data
numerical_features = df[['num_cmc', 'num_power', 'num_toughness', 'num_loyalty']].fillna(-999).values

# ADD BOOLEAN FEATURES
# is_game_changer, toughness_is_variable, power_is_variable, is_planeswalker

# 4. Concatenate Text and Tabular Features
# We stack them horizontally: 384 text features + 4 numerical features = 388 total features
X = np.hstack((embeddings, numerical_features))

# Extract our target variable (the high-precision float score we calculated earlier)
y = df['power_score_float'].values

print(f"Final Feature Matrix (X) ready! Shape: {X.shape}")
print(f"Target Variable (y) ready! Shape: {y.shape}")

In [ ]:
# TODO: think of boolean features (maybe change is_modal, add is_transform, is_saga?, is_class?)
# https://scryfall.com/card/mid/327/enduring-angel-angelic-enforcer?random=%2A&unique=cards

print("Engineering boolean features...")

# --- 1. User Requested Features ---

# Handle the existing game_changer boolean (convert True/False to 1/0)
df['bool_is_game_changer'] = df['game_changer'].fillna(False).astype(int)

# Check for variable power/toughness (looks for the literal '*' or 'X' characters)
df['bool_power_var'] = df['power'].astype(str).str.contains(r'[\*X]', na=False).astype(int)
df['bool_toughness_var'] = df['toughness'].astype(str).str.contains(r'[\*X]', na=False).astype(int)

# Check if it's a Planeswalker based on the type line
df['bool_is_planeswalker'] = df['type_line'].astype(str).str.contains('Planeswalker', case=False, na=False).astype(int)


# --- 2. Recommended Features ---

# Check if the mana cost contains {X}
df['bool_has_x_cost'] = df['mana_cost'].astype(str).str.contains('{X}', case=False, na=False).astype(int)

# Check if the card is Legendary
df['bool_is_legendary'] = df['type_line'].astype(str).str.contains('Legendary', case=False, na=False).astype(int)

# Check if it is colorless (assuming 'color_identity' is a list like ['W', 'U'] or an empty list [] for colorless)
# We check if the length is 0, or if it's the string representation of an empty list
df['bool_is_colorless'] = df['color_identity'].apply(
    lambda x: 1 if (isinstance(x, list) and len(x) == 0) or str(x) == '[]' else 0
)

# Check for modal choices (bullet points in oracle text) or split cards (card_faces list)
df['bool_is_modal'] = df.apply(
    lambda row: 1 if isinstance(row.get('card_faces'), list) or '•' in str(row.get('oracle_text', '')) else 0, 
    axis=1
)


# --- 3. Combine Everything for the Model ---

# Extract the booleans into a NumPy array
boolean_cols = [
    'bool_is_game_changer', 'bool_power_var', 'bool_toughness_var', 
    'bool_is_planeswalker', 'bool_has_x_cost', 'bool_is_legendary', 
    'bool_is_colorless', 'bool_is_modal'
]
boolean_features = df[boolean_cols].values

# Extract the numericals (from your previous code)
numerical_cols = ['num_cmc', 'num_power', 'num_toughness', 'num_loyalty']
numerical_features = df[numerical_cols].fillna(-999).values

# Stack Text Embeddings (384) + Numericals (4) + Booleans (8) = 396 total features!
X = np.hstack((embeddings, numerical_features, boolean_features))

print(f"Final Feature Matrix (X) ready! Shape: {X.shape}")

_Consider calculating strength per type_